In [0]:
# Union history and live tables
# Perform the group by the hour

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_hourlyprices_last90days(
  datetime TIMESTAMP,
  prices DOUBLE,
  market_caps DOUBLE
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/history/gold_hourlyprices_last90days/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_5minprices_lastday(
  datetime TIMESTAMP,
  prices DOUBLE,
  market_caps DOUBLE
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/live/gold_5minprices_lastday/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_4hourlyohlc_last30days(
  datetime TIMESTAMP,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/history/gold_4hourlyohlc_last30days/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_30minohlc_lastday(
  datetime TIMESTAMP,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE
)
USING DELTA
LOCATION 'abfss://gold@bitcoindatalake.dfs.core.windows.net/live/gold_30minohlc_lastday/'

In [0]:
%sql
MERGE INTO gold_hourlyprices_last90days as target
USING(
SELECT *
FROM bitcoin_hourlyprices_last90days
UNION ALL
SELECT hour_start as datetime, prices, market_caps
FROM(
    SELECT
        DATE_TRUNC('hour', datetime) AS hour_start,
        AVG(prices) AS prices,
        AVG(market_caps) as market_caps
    FROM
        bitcoin_5minprices_lastday
    GROUP BY
        hour_start
    ORDER BY
        hour_start
)
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
MERGE INTO gold_4hourlyohlc_last30days as target
USING(
SELECT *
FROM bitcoin_4hourlyohlc_last30days
UNION ALL
SELECT 
window.start as datetime,
First(open) as open,
MAX(high) as high,
MIN(low) as low,
Last(close) as close
FROM bitcoin_30minohlc_lastday
WHERE datetime >= date_trunc('day', current_timestamp())
GROUP BY window(datetime, '4 hours')
ORDER BY datetime
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
MERGE INTO gold_5minprices_lastday as target
USING(
SELECT *
FROM bitcoin_5minprices_lastday
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
MERGE INTO gold_30minohlc_lastday as target
USING(
SELECT *
FROM bitcoin_30minohlc_lastday
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *